# Parametre ve Memory Optimizasyonu

Bu notebook, model parametrelerinin bellekte nasıl saklandığını ve memory optimizasyon tekniklerini incelemektedir.

## 1. Gerekli Kütüphaneler

In [2]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 60)
print("PARAMETRE VE MEMORY OPTİMİZASYONU")
print("=" * 60)

PARAMETRE VE MEMORY OPTİMİZASYONU


## 2. PyTorch Veri Tipleri

In [3]:
print("\n--- PYTORCH VERİ TİPLERİ ---")

dtypes_info = {
    "float32": 4,
    "float16": 2,
    "bfloat16": 2,
    "int8": 1
}

print(f"{'Dtype':<15} {'Bytes':<10}")
print("-" * 25)
for dtype, bytes_size in dtypes_info.items():
    print(f"{dtype:<15} {bytes_size}")


--- PYTORCH VERİ TİPLERİ ---
Dtype           Bytes     
-------------------------
float32         4
float16         2
bfloat16        2
int8            1


## 3. Basit Model ile Parametre Gösterme

In [4]:
# Basit bir linear model
simple_model = nn.Linear(10, 5)

print("\n--- BASİT MODEL PARAMETRELERİ ---")
print(f"Weight shape: {simple_model.weight.shape}")
print(f"Bias shape:   {simple_model.bias.shape}")
print(f"Toplam parametre: {simple_model.weight.numel() + simple_model.bias.numel()}")

print(f"\nDtype: {next(simple_model.parameters()).dtype}")
print(f"Element size: {next(simple_model.parameters()).element_size()} bytes")


--- BASİT MODEL PARAMETRELERİ ---
Weight shape: torch.Size([5, 10])
Bias shape:   torch.Size([5])
Toplam parametre: 55

Dtype: torch.float32
Element size: 4 bytes


## 4. Gradient Saklanması

In [5]:
# Forward ve Backward pass
x = torch.randn(2, 10)
y = simple_model(x)
loss = y.sum()
loss.backward()

print("\n--- GRADIENT SAKLANMASI ---")
for name, param in simple_model.named_parameters():
    print(f"\n{name}:")
    print(f"  Param shape: {param.shape}")
    print(f"  Param bytes: {param.numel() * param.element_size()}")
    print(f"  Grad shape:  {param.grad.shape}")
    print(f"  Grad bytes:  {param.grad.numel() * param.grad.element_size()}")


--- GRADIENT SAKLANMASI ---

weight:
  Param shape: torch.Size([5, 10])
  Param bytes: 200
  Grad shape:  torch.Size([5, 10])
  Grad bytes:  200

bias:
  Param shape: torch.Size([5])
  Param bytes: 20
  Grad shape:  torch.Size([5])
  Grad bytes:  20


## 5. Training vs Inference Memory

In [6]:
# 124M parametre örneği
total_params = 124_000_000

print("\n--- TRAINING vs INFERENCE MEMORY (124M params) ---")

# Inference (sadece model)
inf_fp32 = total_params * 4 / (1024**2)
inf_fp16 = total_params * 2 / (1024**2)

print(f"\nInference:")
print(f"  FP32: {inf_fp32:.1f} MB")
print(f"  FP16: {inf_fp16:.1f} MB")

# Training
train_fp32 = total_params * 4 / (1024**2)  # params
train_fp32 += total_params * 4 / (1024**2)  # gradients
train_fp32 += total_params * 8 / (1024**2)  # AdamW (m + v)

train_fp16 = total_params * 2 / (1024**2)  # params
train_fp16 += total_params * 2 / (1024**2)  # gradients
train_fp16 += total_params * 8 / (1024**2)  # AdamW (FP32)

print(f"\nTraining (AdamW):")
print(f"  FP32: {train_fp32:.1f} MB")
print(f"  FP16 (mixed): {train_fp16:.1f} MB")


--- TRAINING vs INFERENCE MEMORY (124M params) ---

Inference:
  FP32: 473.0 MB
  FP16: 236.5 MB

Training (AdamW):
  FP32: 1892.1 MB
  FP16 (mixed): 1419.1 MB


## 6. Precision Karşılaştırması

In [7]:
print("\n--- FARKLI PRECISION'LAR ---")

precisions = {
    "FP32": {"param": 4, "grad": 4, "opt": 8},
    "FP16": {"param": 2, "grad": 2, "opt": 8},
    "BF16": {"param": 2, "grad": 2, "opt": 8},
}

print(f"\n{'Precision':<10} {'Inference':<15} {'Training':<15}")
print("-" * 40)

for name, bytes_dict in precisions.items():
    inf = total_params * bytes_dict["param"] / (1024**2)
    train = (total_params * bytes_dict["param"] + 
             total_params * bytes_dict["grad"] + 
             total_params * bytes_dict["opt"]) / (1024**2)
    print(f"{name:<10} {inf:>10.0f} MB   {train:>10.0f} MB")


--- FARKLI PRECISION'LAR ---

Precision  Inference       Training       
----------------------------------------
FP32              473 MB         1892 MB
FP16              237 MB         1419 MB
BF16              237 MB         1419 MB


## 7. Memory Hesaplama Fonksiyonu

In [8]:
def estimate_memory(param_count, precision="fp32", train=True, optimizer="adamw"):
    """Model memory gereksinimini hesapla"""
    bytes_dict = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1}
    param_bytes = bytes_dict.get(precision, 4)
    
    # Model parametreleri
    memory = param_count * param_bytes
    
    if train:
        # Gradients
        memory += param_count * param_bytes
        
        # Optimizer state
        if optimizer == "adamw":
            memory += param_count * 8  # m + v (FP32)
        elif optimizer == "sgd":
            memory += param_count * 4  # momentum
    
    return memory / (1024**3)  # GB

# Örnekler
print("\n--- MEMORY HESAPLAMA FONKSİYONU ---")
print(f"124M parametreli model:")
print(f"  Inference FP32: {estimate_memory(124_000_000, 'fp32', train=False):.2f} GB")
print(f"  Training FP32:   {estimate_memory(124_000_000, 'fp32', train=True):.2f} GB")
print(f"  Training FP16:   {estimate_memory(124_000_000, 'fp16', train=True):.2f} GB")
print(f"  Training BF16:   {estimate_memory(124_000_000, 'bf16', train=True):.2f} GB")


--- MEMORY HESAPLAMA FONKSİYONU ---
124M parametreli model:
  Inference FP32: 0.46 GB
  Training FP32:   1.85 GB
  Training FP16:   1.39 GB
  Training BF16:   1.39 GB


## 8. Optimizer State

In [9]:
print("\n--- OPTIMIZER STATE ---")

# AdamW
optimizer = torch.optim.AdamW(simple_model.parameters(), lr=1e-4)

print("AdamW optimizer state (her parametre için):")
first_param = next(simple_model.parameters())
state = optimizer.state[first_param]

for k, v in state.items():
    print(f"  {k}: {v.shape if hasattr(v, 'shape') else 'scalar'}")

print(f"\nEk state parametre sayısı: {sum(v.numel() for v in state.values())}")
print(f"Ek memory (FP32): {sum(v.numel() for v in state.values()) * 4 / 1024:.1f} KB")


--- OPTIMIZER STATE ---
AdamW optimizer state (her parametre için):

Ek state parametre sayısı: 0
Ek memory (FP32): 0.0 KB


## 9. Özet

In [10]:
print("\n" + "=" * 60)
print("ÖZET")
print("=" * 60)

summary = """
┌─────────────────────────────────────────────────────────┐
│ Teknik                │ Memory Azaltma │ Kullanım    │
├───────────────────────┼────────────────┼─────────────┤
│ float16/BF16          │ 2x            │ Çok Kolay   │
│ Mixed Precision       │ 2x            │ Kolay       │
│ Gradient Checkpoint   │ 1.5-2x        │ Orta        │
│ Quantization (INT8)  │ 4x            │ Orta        │
│ LoRA Fine-tuning     │ 10x+          │ Kolay       │
└───────────────────────┴────────────────┴─────────────┘
"""
print(summary)

print("📌 124M Model Örneği:")
print(f"   Inference (FP32): ~{estimate_memory(124_000_000, 'fp32', train=False):.1f} GB")
print(f"   Training (FP32):  ~{estimate_memory(124_000_000, 'fp32', train=True):.1f} GB")
print(f"   Training (FP16):  ~{estimate_memory(124_000_000, 'fp16', train=True):.1f} GB")


ÖZET

┌─────────────────────────────────────────────────────────┐
│ Teknik                │ Memory Azaltma │ Kullanım    │
├───────────────────────┼────────────────┼─────────────┤
│ float16/BF16          │ 2x            │ Çok Kolay   │
│ Mixed Precision       │ 2x            │ Kolay       │
│ Gradient Checkpoint   │ 1.5-2x        │ Orta        │
│ Quantization (INT8)  │ 4x            │ Orta        │
│ LoRA Fine-tuning     │ 10x+          │ Kolay       │
└───────────────────────┴────────────────┴─────────────┘

📌 124M Model Örneği:
   Inference (FP32): ~0.5 GB
   Training (FP32):  ~1.8 GB
   Training (FP16):  ~1.4 GB
